## 10.1 단계별 체인 정의

In [38]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="mistral", temperature=0.1, max_tokens=256)

# 1. 주어진 문서를 한국어로 번역하라
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 스페인어 텍스트를 한국어로 번역하세요."),
    ("user", "{input_text}")
])
translate_chain = translate_prompt | llm | StrOutputParser()

# 2. 번역된 문장에서 핵심 문장을 추출하라.
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 문서에서 핵심 문장을 추출하라."),
    ("user", "{translated_text}")
])
extract_chain = extract_prompt | llm | StrOutputParser()

# 3. 주어진 핵심 문장을 연결하여 요약된 문서를 생성하라.
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 핵심 문장을 연결하여 요약된 문서를 생성하라."),
    ("user", "{key_sentences}")
])
summary_chain = summary_prompt | llm | StrOutputParser()

## 10.2 랭체인을 통한 체이닝 정의 및 실행

In [49]:
# LCEL을 활용한 체이닝 (파이프 연산자 사용)
full_chain = (
    translate_chain  # 1. 번역 실행
    | (lambda translated_text: {"translated_text": translated_text})  # 변환된 텍스트를 다음 단계에 전달
    | extract_chain  # 2. 핵심 문장 추출 실행
    | (lambda key_sentences: {"key_sentences": key_sentences})  # 추출된 문장을 다음 단계에 전달
    | summary_chain  # 3. 최종 요약 실행
)

# 테스트 실행
spanish_text = """España es un país situado en el suroeste de Europa. Su capital es Madrid y su idioma oficial es el español. Tiene una población de aproximadamente 47 millones de personas. España es conocida por su historia, cultura y gastronomía, incluyendo la paella y el flamenco. Además, es un destino turístico popular con ciudades como Barcelona, Sevilla y Valencia."""
result = full_chain.invoke({"input_text": spanish_text})

# 결과 출력
print(result)

스페인은 유럽 남서부에 위치한 국가이며, 공식 언어는 스페인어입니다. 수도는 마드리드이며, 약 4700만 명의 인구를 보유하고 있습니다. 역사, 문화, 요리 등에서 유명한 국가이며, 알려져 있는 것들 중에는 파엘라와 플래멘코와 같은 것들이 있습니다. 또한 투어 목적으로 인기 있는 도시 중 하나로, 바르셀로나, 세비ль라와 발런시아를 포함하여 등장합니다.


## 10.3 각 단계별 체인의 입력과 출력 값

In [50]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import SystemMessage

# 모델의 입력과 출력 튜플을 담을 배열
model_io = []

# 최초 입력 값
step_input = {"input_text": spanish_text}

# 체인을 단계별로 순회
for step in full_chain.steps:
    # 단계별 실행
    step_output = step.invoke(step_input)
    # 모델의 입력과 출력 값만 기록
    if isinstance(step, BaseChatModel):
        model_io.append((step_input, step_output))
    # 현재 단계의 출력을 다음 단계의 입력으로 정의
    step_input = step_output
    
# 모델 입력과 출력을 보기 좋게 출력한다
for input_data, output_data in model_io:
    print("=" * 80)
    print("📌 [입력 메시지]")
    for message in input_data.messages:
        role = "시스템" if isinstance(message, SystemMessage) else "사용자"
        content = message.content if role == "시스템" else message.content
        print(f"▶ {role}: {content}")
    print("\n📝 [출력 결과]")
    print(output_data.content)
    print("=" * 80)

## 10.4 Langgraph 라이브러리 설치

In [2]:
!pip install langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.1/420.1 kB 5.8 MB/s eta 0:00:00m eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.9/351.9 kB 4.9 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.8/382.8 kB 5.6 MB/s eta 0:00:000m eta 0:00:01
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.0.87
    Uninstalling langsmith-0.0.87:
      Successfully uninstalled langsmith-0.0.87
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.1.23
    Uninstalling langchain-core-0.1.23:
      Successfully uninstalled langchain-core-0.1.23
ERROR: pip's dependency resolver does not 

## 10.5 도구 정의

In [ ]:
from langchain_core.tools import tool

# 날씨 정보 조회 도구 정의
@tool
def get_weather(location: str):
    """요청한 지역의 날씨 정보를 찾습니다"""
    if location in ("서울", "seoul"):
        return "오늘 서울 날씨는 따뜻합니다."
    if location in ("부산", "busan"):
        return "오늘 부산 날씨는 흐립니다."
    return "죄송합니다. 요청하신 지역의 날씨 정보를 제공할 수 없습니다."

# 맛집 정보 조회 도구 정의
@tool
def get_good_restaurant(location: str):
    """요청한 지역의 맛집 정보를 찾습니다"""
    if location in ("서울", "seoul"):
        return "서울에서 가장 유명한 맛집은 광장시장, 홍대식당, 백리향입니다."
    if location in ("부산", "busan"):
        return "부산에서 가장 유명한 맛집은 자갈치시장과 민락수산입니다."
    return "죄송합니다. 요청하신 지역의 맛집 정보를 제공할 수 없습니다."


## 10.6 LangGraph 기반 에이전트 정의

In [5]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_anthropic import ChatAnthropic

# 에이전트에 전달할 도구 목록
tools = [get_weather, get_good_restaurant]

# 모델 정의
model = ChatAnthropic(model='claude-3-5-sonnet-20241022', temperature=0.1, api_key=API_KEY)

# 이전 대화 내용을 저장할 메모리 정의
checkpointer = MemorySaver()

# 에이전트 애플리케이션 정의
app = create_react_agent(model, tools, checkpointer=checkpointer)

종합하면, 오늘은 서울의 날씨가 따뜻하고 부산은 흐린 날씨이므로 서울이 더 좋습니다. 서울에 가시면 광장시장, 홍대식당, 백리향과 같은 유명한 맛집들을 방문해보시는 것을 추천드립니다!


## 10.7 에이전트를 통한 응답 예시

In [ ]:
question = "서울과 부산 중에 날씨가 좋은 지역을 찾아줘. 그리고 그 지역의 맛집도 알려줘."
final_state = app.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"configurable": {"thread_id": 1}}
)

print(final_state["messages"][-1].content)

## 10.8 에이전트의 처리 과정

In [40]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage
)

def get_role(message_type):
    if isinstance(message, HumanMessage):
        return "사용자"
    if isinstance(message, AIMessage):
        return "AI"
    if isinstance(message, ToolMessage):
        return "도구"
    
for message in final_state["messages"]:
    role = get_role(message)
    print(f"▶ {role}:{message.content}")

▶ 사용자:서울과 부산 중에 날씨가 좋은 지역을 찾아줘. 그리고 그 지역의 맛집도 알려줘.
▶ AI:[{'text': '네, 서울과 부산의 날씨를 확인하고 비교한 후, 날씨가 더 좋은 지역의 맛집을 알려드리겠습니다.\n\n먼저 두 지역의 날씨를 확인해보겠습니다:', 'type': 'text'}, {'id': 'toolu_01JvLHCELSo9P9xWwBZkpLp4', 'input': {'location': '서울'}, 'name': 'get_weather', 'type': 'tool_use'}]
▶ 도구:오늘 서울 날씨는 따뜻합니다.
▶ AI:[{'id': 'toolu_01RWUxPvfEJsu5mrCFVtqvhX', 'input': {'location': '부산'}, 'name': 'get_weather', 'type': 'tool_use'}]
▶ 도구:오늘 부산 날씨는 흐립니다.
▶ AI:[{'text': '서울이 부산보다 날씨가 더 좋네요! 따라서 서울의 맛집을 알아보겠습니다:', 'type': 'text'}, {'id': 'toolu_015MdGuz3C5LsE4NNq8usukx', 'input': {'location': '서울'}, 'name': 'get_good_restaurant', 'type': 'tool_use'}]
▶ 도구:서울에서 가장 유명한 맛집은 광장시장, 홍대식당, 백리향입니다.
▶ AI:종합하면, 오늘은 서울이 부산보다 날씨가 더 좋습니다. 서울에 방문하신다면 광장시장, 홍대식당, 백리향과 같은 유명한 맛집들을 방문해보시는 것을 추천드립니다!
